Wizualizacja 1: grf vs. white noise

In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath('..'))

from utils.math_functions import MathFunctions
from edm1d.edmdenoiser1d import generate_grf_1d

mf = MathFunctions(num_points=128)
functions_to_plot = ['sin', 'square_wave', '1_over_x']
x_vals = mf.x_std

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle("Porównanie wpływu białego szumu i szumu GRF na wykresy funkcji 1D", fontsize=18, fontweight='bold', y=1.02)

noise_level = 0.5
torch.manual_seed(42)

for i, func_name in enumerate(functions_to_plot):
    _, y_true = mf.get_function(func_name)
    y_tensor = torch.tensor(y_true, dtype=torch.float32).unsqueeze(0)
    
    white_noise = torch.randn_like(y_tensor)
    grf_noise = generate_grf_1d((1, 128), sigma_kernel=2.0).unsqueeze(0)
    
    y_white = y_tensor + noise_level * white_noise
    y_grf = y_tensor + noise_level * grf_noise
    
    axes[i, 0].plot(x_vals, y_true, color='#2ca02c', lw=3)
    axes[i, 0].set_title(f"Ground Truth: {func_name.upper()}")
    axes[i, 0].grid(True, alpha=0.3)
    
    axes[i, 1].plot(x_vals, y_white.squeeze().numpy(), color='gray', alpha=0.6, label='Zaszumiony')
    axes[i, 1].plot(x_vals, y_true, color='#2ca02c', lw=2, label='Ground Truth')
    axes[i, 1].set_title(f"Biały szum (White Noise)")
    axes[i, 1].grid(True, alpha=0.3)
    
    axes[i, 2].plot(x_vals, y_grf.squeeze().numpy(), color='#d62728', alpha=0.6, label='Zaszumiony (GRF)')
    axes[i, 2].plot(x_vals, y_true, color='#2ca02c', lw=2, label='Ground Truth')
    axes[i, 2].set_title(f"Gładki szum (GRF)")
    axes[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../wizualizacje/grf_vs_white.png", bbox_inches='tight')
plt.close()

Wizualizacja 2: linear vs. cosine schedule

In [2]:
def get_beta_schedule_local(schedule_type, beta_start=1e-4, beta_end=0.02, n_T=1000):
    if schedule_type == 'linear':
        return np.linspace(beta_start, beta_end, n_T)
    elif schedule_type == 'cosine':
        steps = n_T + 1
        x = np.linspace(0, n_T, steps)
        alphas_cumprod = np.cos(((x / n_T) + 0.008) / 1.008 * np.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return np.clip(betas, 0.0001, 0.9999)

n_T = 1000
betas_linear = get_beta_schedule_local('linear', n_T=n_T)
betas_cosine = get_beta_schedule_local('cosine', n_T=n_T)

alphas_bar_linear = np.cumprod(1. - betas_linear)
alphas_bar_cosine = np.cumprod(1. - betas_cosine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(betas_linear, label='Linear Schedule', lw=2)
axes[0].plot(betas_cosine, label='Cosine Schedule', lw=2)
axes[0].set_title(r"Wartość $\beta_t$ (Ilość szumu dodawana w kroku)")
axes[0].set_xlabel("Krok dyfuzji $t$")
axes[0].set_ylim(0, 0.03) 

axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left')
axes[1].plot(alphas_bar_linear, label='Linear Schedule', lw=2)
axes[1].plot(alphas_bar_cosine, label='Cosine Schedule', lw=2)
axes[1].set_title(r"Wartość $\bar{\alpha}_t$ (Ilość zachowanego sygnału oryginalnego)")
axes[1].set_xlabel("Krok dyfuzji $t$")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.savefig("../wizualizacje/linear_vs_cosine.png",bbox_inches='tight')
plt.close()

Wizualizacja 3: zaszumienie - proces Forward

In [3]:
target_T = 200
timesteps_to_show = [0, 20, 50, 100, 199]

betas = get_beta_schedule_local('cosine', n_T=target_T)
alphas = 1. - betas
alphas_bar = np.cumprod(alphas)

_, y_clean = mf.get_function('sin')
y_clean_tensor = torch.tensor(y_clean, dtype=torch.float32)

fig, axes = plt.subplots(1, 5, figsize=(25, 4))
fig.suptitle("Proces Forward (harmonogram cosine) w czasie $t$", fontsize=16, fontweight='bold', y=1.05)

torch.manual_seed(42)
noise = torch.randn_like(y_clean_tensor)

for i, t in enumerate(timesteps_to_show):
    if t == 0:
        x_t = y_clean_tensor
    else:
        a_bar = alphas_bar[t]
        x_t = np.sqrt(a_bar) * y_clean_tensor + np.sqrt(1 - a_bar) * noise
        
    axes[i].plot(x_vals, x_t.numpy(), color='#1f77b4' if t==0 else 'gray')
    axes[i].set_title(f"Krok t = {t}")
    axes[i].set_ylim(-3, 3)
    axes[i].grid(True, alpha=0.3)

plt.savefig("../wizualizacje/cosine_forward.png", bbox_inches='tight')
plt.close()

betas = get_beta_schedule_local('linear', n_T=target_T)
alphas = 1. - betas
alphas_bar = np.cumprod(alphas)

_, y_clean = mf.get_function('sin')
y_clean_tensor = torch.tensor(y_clean, dtype=torch.float32)

fig, axes = plt.subplots(1, 5, figsize=(25, 4))
fig.suptitle("Proces Forward (harmonogram linear) w czasie $t$", fontsize=16, fontweight='bold', y=1.05)

torch.manual_seed(42)
noise = torch.randn_like(y_clean_tensor)

for i, t in enumerate(timesteps_to_show):
    if t == 0:
        x_t = y_clean_tensor
    else:
        a_bar = alphas_bar[t]
        x_t = np.sqrt(a_bar) * y_clean_tensor + np.sqrt(1 - a_bar) * noise
        
    axes[i].plot(x_vals, x_t.numpy(), color='#1f77b4' if t==0 else 'gray')
    axes[i].set_title(f"Krok t = {t}")
    axes[i].set_ylim(-3, 3)
    axes[i].grid(True, alpha=0.3)

plt.savefig("../wizualizacje/linear_forward.png",bbox_inches='tight')
plt.close()

Wizualizacja 4: 

In [4]:
from edm1d.edmdenoiser1d import FunDPSSampler, FunDPSExperimentRunner, ForwardOperator
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from utils.math_functions import MathFunctions


def generate_fundps_gif(history, y_true, mask_idx, obs_tensor, zeta_val, filename="fundps_evolution.gif"):
    fig, ax = plt.subplots(figsize=(10, 6)) 
    x_axis = np.arange(len(y_true))
    ax.plot(x_axis, y_true, color='gray', linestyle='--', alpha=0.6, label='Prawdziwy sygnał (Ground Truth)')
    measurements = obs_tensor.cpu().numpy()[0]
    ax.scatter(mask_idx, measurements, color='red', s=50, zorder=5, label='Pomiary (Measurements)')
    line, = ax.plot([], [], color='blue', linewidth=2, label='Generowana trajektoria')
    ax.set_xlim(0, len(y_true) - 1)
    all_vals = np.concatenate(history)
    y_min, y_max = np.min(all_vals), np.max(all_vals)
    y_min_disp = max(y_min, np.min(y_true) - 2.0)
    y_max_disp = min(y_max, np.max(y_true) + 2.0)
    ax.set_ylim(y_min_disp, y_max_disp)
    
    ax.legend(loc="upper right")
    ax.grid(True, linestyle=':', alpha=0.6)

    def init():
        line.set_data([], [])
        return line,

    def update(frame):
        current_trajectory = history[frame]
        line.set_data(x_axis, current_trajectory)
        
        if frame == 0:
            ax.set_title(rf"Początkowy szum $\sigma_{{max}}$ | $\zeta$ = {zeta_val}", fontsize=14)
        else:
            ax.set_title(rf"FunDPS: Krok odszumiania: {frame}/{len(history)-1} | $\zeta$ = {zeta_val}", fontsize=14)
        return line,

    ani = animation.FuncAnimation(
        fig, update, frames=len(history),
        init_func=init, blit=True, interval=80 
    )
    
    ani.save(filename, writer='pillow', fps=15)
    plt.close(fig)
    print(f"Zapisano animację jako: {filename}")

In [5]:
func_name = 'sin'  
runner = FunDPSExperimentRunner(noise_type='white') 
x, y_true = runner.math_funcs.get_function(func_name)
y_tensor = torch.tensor(y_true, dtype=torch.float32).unsqueeze(0).to(runner.device)

np.random.seed(42) 
num_obs = int(128 * 0.10)
mask_idx = np.sort(np.random.choice(128, num_obs, replace=False))
forward_op = ForwardOperator(mask_idx)
obs_tensor = forward_op(y_tensor)

print(f"Trenowanie modelu dla funkcji '{func_name}'...")
model, _, _ = runner.train_unconditional_prior(y_tensor, epochs=1000)

sampler = FunDPSSampler(model, runner.device)
steps_for_gif = 100
zeta_for_gif = 200.0  

trajectory_history = sampler.sample_with_history(
    observation=obs_tensor, 
    forward_op=forward_op, 
    num_steps=steps_for_gif, 
    zeta=zeta_for_gif
)

generate_fundps_gif(
    history=trajectory_history, 
    y_true=y_true, 
    mask_idx=mask_idx, 
    obs_tensor=obs_tensor, 
    zeta_val=zeta_for_gif,
    filename="../wizualizacje/fundps_animacja_sin_white.gif"
)


Trenowanie modelu dla funkcji 'sin'...
Zapisano animację jako: ../wizualizacje/fundps_animacja_sin_white.gif


In [6]:
func_name = 'sin'  
runner = FunDPSExperimentRunner(noise_type='grf') 
x, y_true = runner.math_funcs.get_function(func_name)
y_tensor = torch.tensor(y_true, dtype=torch.float32).unsqueeze(0).to(runner.device)

np.random.seed(42) 
num_obs = int(128 * 0.10)
mask_idx = np.sort(np.random.choice(128, num_obs, replace=False))
forward_op = ForwardOperator(mask_idx)
obs_tensor = forward_op(y_tensor)

print(f"Trenowanie modelu dla funkcji '{func_name}'...")
model, _, _ = runner.train_unconditional_prior(y_tensor, epochs=1000)

sampler = FunDPSSampler(model, runner.device)
steps_for_gif = 100
zeta_for_gif = 200.0  

trajectory_history = sampler.sample_with_history(
    observation=obs_tensor, 
    forward_op=forward_op, 
    num_steps=steps_for_gif, 
    zeta=zeta_for_gif
)

generate_fundps_gif(
    history=trajectory_history, 
    y_true=y_true, 
    mask_idx=mask_idx, 
    obs_tensor=obs_tensor, 
    zeta_val=zeta_for_gif,
    filename="../wizualizacje/fundps_animacja_sin_grf.gif"
)


Trenowanie modelu dla funkcji 'sin'...
Zapisano animację jako: ../wizualizacje/fundps_animacja_sin_grf.gif


In [7]:
func_name = '1_over_x'  
runner = FunDPSExperimentRunner(noise_type='grf') 
x, y_true = runner.math_funcs.get_function(func_name)
y_tensor = torch.tensor(y_true, dtype=torch.float32).unsqueeze(0).to(runner.device)

np.random.seed(42) 
num_obs = int(128 * 0.10)
mask_idx = np.sort(np.random.choice(128, num_obs, replace=False))
forward_op = ForwardOperator(mask_idx)
obs_tensor = forward_op(y_tensor)

print(f"Trenowanie modelu dla funkcji '{func_name}'...")
model, _, _ = runner.train_unconditional_prior(y_tensor, epochs=1000)

sampler = FunDPSSampler(model, runner.device)
steps_for_gif = 100
zeta_for_gif = 200.0  

trajectory_history = sampler.sample_with_history(
    observation=obs_tensor, 
    forward_op=forward_op, 
    num_steps=steps_for_gif, 
    zeta=zeta_for_gif
)

generate_fundps_gif(
    history=trajectory_history, 
    y_true=y_true, 
    mask_idx=mask_idx, 
    obs_tensor=obs_tensor, 
    zeta_val=zeta_for_gif,
    filename="../wizualizacje/fundps_animacja_1_over_x_grf.gif"
)


Trenowanie modelu dla funkcji '1_over_x'...
Zapisano animację jako: ../wizualizacje/fundps_animacja_1_over_x_grf.gif


In [8]:
func_name = '1_over_x'  
runner = FunDPSExperimentRunner(noise_type='white') 
x, y_true = runner.math_funcs.get_function(func_name)
y_tensor = torch.tensor(y_true, dtype=torch.float32).unsqueeze(0).to(runner.device)

np.random.seed(42) 
num_obs = int(128 * 0.10)
mask_idx = np.sort(np.random.choice(128, num_obs, replace=False))
forward_op = ForwardOperator(mask_idx)
obs_tensor = forward_op(y_tensor)

print(f"Trenowanie modelu dla funkcji '{func_name}'...")
model, _, _ = runner.train_unconditional_prior(y_tensor, epochs=1000)

sampler = FunDPSSampler(model, runner.device)
steps_for_gif = 100
zeta_for_gif = 200.0  

trajectory_history = sampler.sample_with_history(
    observation=obs_tensor, 
    forward_op=forward_op, 
    num_steps=steps_for_gif, 
    zeta=zeta_for_gif
)

generate_fundps_gif(
    history=trajectory_history, 
    y_true=y_true, 
    mask_idx=mask_idx, 
    obs_tensor=obs_tensor, 
    zeta_val=zeta_for_gif,
    filename="../wizualizacje/fundps_animacja_1_over_x_white.gif"
)


Trenowanie modelu dla funkcji '1_over_x'...
Zapisano animację jako: ../wizualizacje/fundps_animacja_1_over_x_white.gif


Wizualizacja 5: 

In [9]:
import matplotlib.pyplot as plt
import numpy as np

def plot_denoising_evolution(history, y_true, mask_idx, obs_tensor, num_snapshots=5):
    fig, axes = plt.subplots(1, num_snapshots, figsize=(20, 4), sharey=True)
    
    total_frames = len(history)
    step_indices = np.linspace(0, total_frames - 1, num_snapshots, dtype=int)
    
    x_axis = np.arange(len(y_true))
    measurements = obs_tensor.cpu().numpy()[0]

    y_min = max(np.min(history[-1]) - 5.0, -5.0) 
    y_max = min(np.max(history[-1]) + 5.0, 5.0)
    
    fig.suptitle('Proces FunDPS', fontsize=16, fontweight='bold', y=1.05)

    for ax, step_idx in zip(axes, step_indices):
        current_trajectory = history[step_idx]
        
        ax.plot(x_axis, y_true, color='gray', linestyle='--', alpha=0.5, 
                label='Oryginał' if step_idx == 0 else "")
        
        ax.scatter(mask_idx, measurements, color='red', zorder=5, s=40,
                   label='Pomiary' if step_idx == 0 else "")
        
        ax.plot(x_axis, current_trajectory, color='#1f77b4', linewidth=2, 
                label='Generowany sygnał' if step_idx == 0 else "")
        
        progress = (step_idx / (total_frames - 1)) * 100
        
        ax.set_title(f"Krok: {step_idx} ({progress:.0f}%)", fontsize=13)
        ax.set_ylim(y_min, y_max)
        ax.grid(True, linestyle=':', alpha=0.7)
        
    axes[0].legend(loc='lower left', fontsize=10)
    
    plt.tight_layout()
    plt.savefig("../wizualizacje/ewolucja_odszumiania.png", bbox_inches='tight')
    plt.close()

plot_denoising_evolution(
    history=trajectory_history, 
    y_true=y_true, 
    mask_idx=mask_idx, 
    obs_tensor=obs_tensor, 
    num_snapshots=6  
)

# MLP

In [10]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from ddpm1d.ddpm1d_mlp import DDPM1D, DenoiseNet1D_MLP

device = torch.device("cpu")
n_T = 100
betas = np.linspace(1e-4, 0.02, n_T)

# Inicjujemy modele 
model_mlp = DenoiseNet1D_MLP(data_dim=128, time_emb_dim=64).to(device)
ddpm_model = DDPM1D(model=model_mlp, betas=betas, n_T=n_T, device=device)

# Tworzymy czysty sygnał referencyjny (sinusoida)
x_axis = np.linspace(0, 4 * np.pi, 128)
clean_signal = torch.tensor(np.sin(x_axis), dtype=torch.float32).unsqueeze(0).to(device)
# Zamrażamy "losowy" szum, żeby przy przesuwaniu suwaka widzieć ewolucję na tym samym szumie
fixed_noise = torch.randn_like(clean_signal)

# 2. FUNKCJA RYSUJĄCA WNĘTRZE SIECI DLA DANEGO KROKU 't'
def plot_network_internals(t_val):
    model_mlp.eval()
    
    with torch.no_grad():
        # A) PRZYGOTOWANIE WEJŚCIA DLA KONKRETNEGO t
        t_tensor = torch.tensor([t_val], device=device).long()
        alpha_bar_t = ddpm_model.alphas_bar[t_tensor].view(-1, 1)
        
        if t_val == 0:
            x_input = clean_signal
        else:
            # Proces wprzód: zaszumiamy sygnał dla danego kroku t
            x_input = torch.sqrt(alpha_bar_t) * clean_signal + torch.sqrt(1 - alpha_bar_t) * fixed_noise
            
        # B) RĘCZNY PRZEPŁYW PRZEZ WARSTWY (Forward Pass)
        t_emb = model_mlp.time_mlp(t_tensor)                 # Osadzenie czasu
        
        x_fc1 = model_mlp.fc1(x_input)                       # Sygnał -> Warstwa 1
        t_proj1 = model_mlp.time_proj1(t_emb)                # Czas -> Warstwa 1
        x_fused1 = model_mlp.act(x_fc1 + t_proj1)            # FUZJA 1
        
        x_fc2 = model_mlp.fc2(x_fused1)                      # Sygnał -> Warstwa 2
        t_proj2 = model_mlp.time_proj2(t_emb)                # Czas -> Warstwa 2
        x_fused2 = model_mlp.act(x_fc2 + t_proj2)            # FUZJA 2
        
        out = model_mlp.out(x_fused2)                        # Wyjście

    # C) WIZUALIZACJA
    tensors_to_plot = {
        f"1. Wejście: Zaszumiony Sygnał w t={t_val} (128 neuronów)": x_input,
        f"2. Osadzenie Czasu t={t_val} (64 neurony)": t_emb,
        "3. Sygnał w Warstwie FC1 (256 neuronów)": x_fc1,
        "4. Czas rzutowany na Warstwę 1 (256 neuronów)": t_proj1,
        "5. FUZJA 1: Sygnał + Czas (256 neuronów)": x_fused1,
        "6. FUZJA 2: Druga warstwa ukryta (256 neuronów)": x_fused2,
        "7. Wyjście: Przewidziany Szum (128 neuronów)": out
    }

    fig, axes = plt.subplots(len(tensors_to_plot), 1, figsize=(10, 8))
    fig.suptitle(f"Wnętrze DenoiseNet1D_MLP | Krok czasowy t = {t_val}", fontsize=14, fontweight='bold')

    for i, (title, tensor) in enumerate(tensors_to_plot.items()):
        ax = axes[i]
        data = tensor.cpu().numpy()
        
        im = ax.imshow(data, aspect='auto', cmap='magma')
        ax.set_title(title, fontsize=10, loc='left', pad=3)
        ax.set_yticks([]) 
        ax.set_xticks([])
        
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# 3. URUCHOMIENIE INTERAKTYWNEGO WIDGETU
t_slider = widgets.IntSlider(
    value=99,          # Wartość startowa
    min=0,             # Minimum
    max=n_T - 1,       # Maksimum
    step=1,            # Krok
    description='Krok t:',
    continuous_update=False
)

interactive_plot = widgets.interactive(plot_network_internals, t_val=t_slider)
display(interactive_plot)

interactive(children=(IntSlider(value=99, continuous_update=False, description='Krok t:', max=99), Output()), …

In [11]:
from IPython.display import display, HTML

html_code = """
<div style="border: 2px solid #ddd; padding: 15px; border-radius: 10px; background-color: #ffffff; font-family: sans-serif; width: 100%; max-width: 800px;">
    
    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px;">
        <div>
            <label style="font-weight: bold;">Krok czasowy t: <span id="t-value">100</span></label><br>
            <input type="range" id="t-slider" min="0" max="100" value="100" style="width: 200px;">
        </div>
        <button id="sim-btn" style="padding: 10px 20px; font-size: 14px; font-weight: bold; cursor: pointer; background-color: #4CAF50; color: white; border: none; border-radius: 5px;">
            ▶ Symuluj przepływ (Forward Pass)
        </button>
    </div>

    <div style="position: relative;">
        <svg id="nn-svg" width="100%" height="450" style="background-color: #f8f9fa; border-radius: 5px; border: 1px solid #eee;"></svg>
    </div>

    <script>
        (function() {
            const svg = document.getElementById('nn-svg');
            const slider = document.getElementById('t-slider');
            const tValueDisplay = document.getElementById('t-value');
            const simBtn = document.getElementById('sim-btn');

            // Definicja warstw (x, y dla każdego neuronu)
            const layers = {
                input: Array.from({length: 6}, (_, i) => ({x: 50, y: 50 + i*40})),
                time: Array.from({length: 3}, (_, i) => ({x: 50, y: 340 + i*30})),
                hidden1: Array.from({length: 5}, (_, i) => ({x: 250, y: 70 + i*40})),
                hidden2: Array.from({length: 5}, (_, i) => ({x: 450, y: 70 + i*40})),
                output: Array.from({length: 6}, (_, i) => ({x: 650, y: 50 + i*40}))
            };

            const connections = [];

            // Funkcja pomocnicza do tworzenia linii (połączeń między neuronami)
            function drawLink(source, target, isTime = false) {
                const line = document.createElementNS("http://www.w3.org/2000/svg", "line");
                line.setAttribute("x1", source.x); line.setAttribute("y1", source.y);
                line.setAttribute("x2", target.x); line.setAttribute("y2", target.y);
                line.setAttribute("stroke", isTime ? "rgba(255, 152, 0, 0.2)" : "rgba(100, 100, 100, 0.15)");
                line.setAttribute("stroke-width", "1.5");
                svg.appendChild(line);
                connections.push({source, target, isTime});
            }

            // Funkcja pomocnicza do rysowania neuronów
            function drawNode(x, y, color, labelText) {
                const circle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
                circle.setAttribute("cx", x); circle.setAttribute("cy", y);
                circle.setAttribute("r", 8);
                circle.setAttribute("fill", color);
                circle.setAttribute("stroke", "#333");
                svg.appendChild(circle);
                return circle;
            }

            // 1. RYSOWANIE POŁĄCZEŃ (krawędzi)
            layers.input.forEach(s => layers.hidden1.forEach(t => drawLink(s, t)));
            layers.time.forEach(s => layers.hidden1.forEach(t => drawLink(s, t, true))); // Czas -> H1
            layers.hidden1.forEach(s => layers.hidden2.forEach(t => drawLink(s, t)));
            layers.time.forEach(s => layers.hidden2.forEach(t => drawLink(s, t, true))); // Czas -> H2
            layers.hidden2.forEach(s => layers.output.forEach(t => drawLink(s, t)));

            // 2. RYSOWANIE NEURONÓW (węzłów)
            const inputNodes = layers.input.map(n => drawNode(n.x, n.y, "#2196F3"));
            const timeNodes = layers.time.map(n => drawNode(n.x, n.y, "#FF9800"));
            layers.hidden1.map(n => drawNode(n.x, n.y, "#9C27B0"));
            layers.hidden2.map(n => drawNode(n.x, n.y, "#9C27B0"));
            layers.output.map(n => drawNode(n.x, n.y, "#4CAF50"));

            // Etykiety tekstowe
            const addText = (x, y, txt) => {
                const t = document.createElementNS("http://www.w3.org/2000/svg", "text");
                t.setAttribute("x", x); t.setAttribute("y", y); t.setAttribute("font-size", "12"); t.textContent = txt;
                svg.appendChild(t);
            };
            addText(20, 25, "Wejście (x)");
            addText(15, 320, "Czas (t_emb)");
            addText(220, 25, "Warstwa Ukryta 1");
            addText(420, 25, "Warstwa Ukryta 2");
            addText(620, 25, "Wyjście (Szum)");

            // 3. REAKCJA NA SUWAK (Miganie neuronów wejściowych od szumu)
            slider.addEventListener('input', (e) => {
                const t = e.target.value;
                tValueDisplay.innerText = t;
                const noiseIntensity = t / 100; 
                
                inputNodes.forEach(node => {
                    // Im większe t, tym bardziej "szaleją" neurony wejściowe
                    const jitterX = 50 + (Math.random() - 0.5) * 10 * noiseIntensity;
                    node.setAttribute("cx", jitterX);
                    node.setAttribute("fill", `rgb(${Math.floor(33 + noiseIntensity*200)}, 150, 243)`);
                });
            });

            // 4. ANIMACJA PRZEPŁYWU (Strzelanie kuleczkami wzdłuż krawędzi)
            simBtn.addEventListener('click', () => {
                simBtn.disabled = true;
                
                connections.forEach((conn, index) => {
                    // Opóźniamy start animacji w zależności od warstwy, by stworzyć "falę"
                    let delay = 0;
                    if (conn.source.x === 250) delay = 600; // start z Hidden1
                    if (conn.source.x === 450) delay = 1200; // start z Hidden2

                    setTimeout(() => {
                        const particle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
                        particle.setAttribute("cx", conn.source.x);
                        particle.setAttribute("cy", conn.source.y);
                        particle.setAttribute("r", 3);
                        particle.setAttribute("fill", conn.isTime ? "#FF9800" : "#2196F3");
                        svg.appendChild(particle);

                        // Używamy Web Animations API do przesunięcia kropki
                        const animation = particle.animate([
                            { transform: `translate(0px, 0px)` },
                            { transform: `translate(${conn.target.x - conn.source.x}px, ${conn.target.y - conn.source.y}px)` }
                        ], {
                            duration: 600,
                            easing: 'ease-in-out'
                        });

                        // Usuwamy kropkę, gdy doleci do celu
                        animation.onfinish = () => particle.remove();
                    }, delay + (Math.random() * 200)); // lekki losowy rozrzut dla naturalnego efektu
                });

                // Odblokuj przycisk po zakończeniu całości
                setTimeout(() => { simBtn.disabled = false; }, 2000);
            });
        })();
    </script>
</div>
"""

display(HTML(html_code))

# Conv1D

In [12]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from ddpm1d.ddpm1d_conv1d import DenoiseNet1D_Conv

device = torch.device("cpu")
n_T = 100
betas = np.linspace(1e-4, 0.02, n_T)

# Inicjalizacja modelu Conv1D
model_conv = DenoiseNet1D_Conv(data_dim=128, time_emb_dim=64, channels=32).to(device)
ddpm_conv = DDPM1D(model=model_conv, betas=betas, n_T=n_T, device=device)

x_axis = np.linspace(0, 4 * np.pi, 128)
clean_signal = torch.tensor(np.sin(x_axis), dtype=torch.float32).unsqueeze(0).to(device)
fixed_noise = torch.randn_like(clean_signal)

def plot_conv1d_internals(t_val):
    model_conv.eval()
    
    with torch.no_grad():
        t_tensor = torch.tensor([t_val], device=device).long()
        alpha_bar_t = ddpm_conv.alphas_bar[t_tensor].view(-1, 1)
        
        if t_val == 0:
            x_input = clean_signal
        else:
            x_input = torch.sqrt(alpha_bar_t) * clean_signal + torch.sqrt(1 - alpha_bar_t) * fixed_noise
            
        # --- RĘCZNY FORWARD PASS DLA CONV1D ---
        x_unsqueezed = x_input.unsqueeze(1) # [1, 1, 128]
        
        t_emb = model_conv.time_mlp(t_tensor) 
        t_feat = model_conv.time_proj(t_emb).unsqueeze(-1) # [1, 32, 1]
        
        x_conv_in = model_conv.conv_in(x_unsqueezed)       # [1, 32, 128]
        
        # Fuzja 1
        x_fused1 = x_conv_in + t_feat
        x_block1 = model_conv.block1(x_fused1)             # [1, 32, 128]
        
        # Fuzja 2
        x_fused2 = x_block1 + t_feat
        x_block2 = model_conv.block2(x_fused2)             # [1, 32, 128]
        
        out = model_conv.conv_out(x_block2).squeeze(1)     # [1, 128]

    # --- WIZUALIZACJA ---
    # Squeeze(0) usuwa wymiar Batch, zostawiając [Kanały, Długość]
    # Rozszerzamy t_feat wizualnie, żeby pokazać jak wpływa na cały sygnał
    t_feat_vis = t_feat.expand(-1, -1, 128)
    
    tensors_to_plot = {
        f"1. Wejście: Sygnał t={t_val} (1x128)": x_input.unsqueeze(0), 
        "2. Conv In: Ekstrakcja 32 kanałów (32x128)": x_conv_in,
        "3. Czas (t_feat) rozszerzony na sygnał (32x128)": t_feat_vis,
        "4. Po Block 1 (Fuzja czasu i konwolucja) (32x128)": x_block1,
        "5. Po Block 2 (Dalsze przetwarzanie) (32x128)": x_block2,
        "6. Wyjście: Przewidziany Szum (1x128)": out.unsqueeze(0)
    }

    fig, axes = plt.subplots(len(tensors_to_plot), 1, figsize=(10, 8), gridspec_kw={'height_ratios': [1, 3, 3, 3, 3, 1]})
    fig.suptitle(f"Wnętrze Conv1D | Krok czasowy t = {t_val}", fontsize=14, fontweight='bold')

    for i, (title, tensor) in enumerate(tensors_to_plot.items()):
        ax = axes[i]
        data = tensor.squeeze(0).cpu().numpy() # Kształt: [Kanały, 128]
        
        im = ax.imshow(data, aspect='auto', cmap='viridis', interpolation='nearest')
        ax.set_title(title, fontsize=10, loc='left', pad=3)
        ax.set_yticks([]) 
        ax.set_xticks([]) 
        
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Widget
t_slider_conv = widgets.IntSlider(value=99, min=0, max=n_T-1, step=1, description='Krok t:', continuous_update=False)
display(widgets.interactive(plot_conv1d_internals, t_val=t_slider_conv))

interactive(children=(IntSlider(value=99, continuous_update=False, description='Krok t:', max=99), Output()), …

In [13]:
from IPython.display import display, HTML

html_code_conv = """
<div style="border: 2px solid #ddd; padding: 15px; border-radius: 10px; background-color: #ffffff; font-family: sans-serif; width: 100%; max-width: 850px;">
    
    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px;">
        <div>
            <label style="font-weight: bold;">Krok czasowy t: <span id="t-value-conv">100</span></label><br>
            <input type="range" id="t-slider-conv" min="0" max="100" value="100" style="width: 200px;">
        </div>
        <button id="sim-btn-conv" style="padding: 10px 20px; font-size: 14px; font-weight: bold; cursor: pointer; background-color: #E91E63; color: white; border: none; border-radius: 5px;">
            ▶ Symuluj Conv1D (Forward Pass)
        </button>
    </div>

    <div style="position: relative;">
        <svg id="nn-svg-conv" width="100%" height="480" style="background-color: #fcfcfc; border-radius: 5px; border: 1px solid #eee;"></svg>
    </div>

    <script>
        (function() {
            const svg = document.getElementById('nn-svg-conv');
            const slider = document.getElementById('t-slider-conv');
            const tValueDisplay = document.getElementById('t-value-conv');
            const simBtn = document.getElementById('sim-btn-conv');

            // Definicja warstw: Wejście (1D), Conv_in (Głębsze), Block1, Block2, Wyjście (1D)
            const layers = {
                input: Array.from({length: 6}, (_, i) => ({x: 50, y: 70 + i*40})),
                time: Array.from({length: 3}, (_, i) => ({x: 50, y: 380 + i*30})),
                
                // Reprezentacja 32 kanałów jako zgrupowane "pudełka" węzłów (3x5)
                convIn: Array.from({length: 15}, (_, i) => ({x: 200 + (i%3)*15, y: 70 + Math.floor(i/3)*40})),
                block1: Array.from({length: 15}, (_, i) => ({x: 400 + (i%3)*15, y: 70 + Math.floor(i/3)*40})),
                block2: Array.from({length: 15}, (_, i) => ({x: 600 + (i%3)*15, y: 70 + Math.floor(i/3)*40})),
                
                output: Array.from({length: 6}, (_, i) => ({x: 780, y: 70 + i*40}))
            };

            const connections = [];

            function drawLink(source, target, isTime = false) {
                const line = document.createElementNS("http://www.w3.org/2000/svg", "line");
                line.setAttribute("x1", source.x); line.setAttribute("y1", source.y);
                line.setAttribute("x2", target.x); line.setAttribute("y2", target.y);
                line.setAttribute("stroke", isTime ? "rgba(255, 87, 34, 0.25)" : "rgba(100, 100, 100, 0.1)");
                line.setAttribute("stroke-width", isTime ? "2" : "1");
                svg.appendChild(line);
                connections.push({source, target, isTime});
            }

            function drawNode(x, y, color, size=6) {
                const circle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
                circle.setAttribute("cx", x); circle.setAttribute("cy", y);
                circle.setAttribute("r", size);
                circle.setAttribute("fill", color);
                circle.setAttribute("stroke", "#333");
                circle.setAttribute("stroke-width", "0.5");
                svg.appendChild(circle);
                return circle;
            }

            // 1. RYSOWANIE POŁĄCZEŃ
            // Input -> Conv_in
            layers.input.forEach(s => layers.convIn.forEach(t => { if(Math.random() > 0.6) drawLink(s, t); }));
            
            // Fuzja 1: Czas dodawany przed Block 1 (x = x + t_feat)
            layers.time.forEach(s => layers.block1.forEach(t => { if(Math.random() > 0.5) drawLink(s, t, true); }));
            layers.convIn.forEach(s => layers.block1.forEach(t => { if(Math.random() > 0.7) drawLink(s, t); }));
            
            // Fuzja 2: Czas dodawany przed Block 2 (x = x + t_feat)
            layers.time.forEach(s => layers.block2.forEach(t => { if(Math.random() > 0.5) drawLink(s, t, true); }));
            layers.block1.forEach(s => layers.block2.forEach(t => { if(Math.random() > 0.7) drawLink(s, t); }));
            
            // Block 2 -> Output
            layers.block2.forEach(s => layers.output.forEach(t => { if(Math.random() > 0.6) drawLink(s, t); }));

            // 2. RYSOWANIE NEURONÓW
            const inputNodes = layers.input.map(n => drawNode(n.x, n.y, "#00BCD4", 8));
            const timeNodes = layers.time.map(n => drawNode(n.x, n.y, "#FF5722", 8));
            layers.convIn.map(n => drawNode(n.x, n.y, "#673AB7"));
            layers.block1.map(n => drawNode(n.x, n.y, "#673AB7"));
            layers.block2.map(n => drawNode(n.x, n.y, "#673AB7"));
            layers.output.map(n => drawNode(n.x, n.y, "#8BC34A", 8));

            // ETYKIETY
            const addText = (x, y, txt, weight="normal") => {
                const t = document.createElementNS("http://www.w3.org/2000/svg", "text");
                t.setAttribute("x", x); t.setAttribute("y", y); 
                t.setAttribute("font-size", "12"); t.setAttribute("font-weight", weight);
                t.textContent = txt;
                svg.appendChild(t);
            };
            
            addText(15, 30, "Wejście", "bold"); addText(15, 45, "(1 Kanał)");
            addText(15, 360, "Czas (t_feat)", "bold");
            addText(185, 30, "conv_in", "bold"); addText(185, 45, "(32 Kanały)");
            addText(385, 30, "block1", "bold"); addText(385, 45, "(32 Kanały)");
            addText(585, 30, "block2", "bold"); addText(585, 45, "(32 Kanały)");
            addText(750, 30, "Wyjście", "bold"); addText(750, 45, "(1 Kanał)");

            // Wskaźniki Fuzji (x = x + t_feat)
            addText(310, 270, "x + t_feat");
            addText(510, 270, "x + t_feat");

            // 3. LOGIKA SUWAKA (Miganie szumu)
            slider.addEventListener('input', (e) => {
                const t = e.target.value;
                tValueDisplay.innerText = t;
                const noiseIntensity = t / 100; 
                
                inputNodes.forEach(node => {
                    const jitterX = 50 + (Math.random() - 0.5) * 12 * noiseIntensity;
                    node.setAttribute("cx", jitterX);
                    node.setAttribute("fill", `rgb(${Math.floor(0 + noiseIntensity*150)}, 188, 212)`);
                });
            });

            // 4. ANIMACJA PRZEPŁYWU
            simBtn.addEventListener('click', () => {
                simBtn.disabled = true;
                
                connections.forEach((conn) => {
                    let delay = 0;
                    if (conn.source.x > 100 && conn.source.x < 300) delay = 800; // start z conv_in
                    if (conn.source.x > 300 && conn.source.x < 500) delay = 1600; // start z block1
                    if (conn.source.x > 500) delay = 2400; // start z block2

                    // Opóźnienie dla strumieni czasu
                    if (conn.isTime) {
                        if (conn.target.x > 300 && conn.target.x < 500) delay = 800; // Czas dolatujący do Block1
                        if (conn.target.x > 500) delay = 1600; // Czas dolatujący do Block2
                    }

                    setTimeout(() => {
                        const particle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
                        particle.setAttribute("cx", conn.source.x);
                        particle.setAttribute("cy", conn.source.y);
                        particle.setAttribute("r", conn.isTime ? 4 : 2);
                        particle.setAttribute("fill", conn.isTime ? "#FF5722" : "#00BCD4");
                        svg.appendChild(particle);

                        const animation = particle.animate([
                            { transform: `translate(0px, 0px)` },
                            { transform: `translate(${conn.target.x - conn.source.x}px, ${conn.target.y - conn.source.y}px)` }
                        ], {
                            duration: 700,
                            easing: 'ease-in-out'
                        });

                        animation.onfinish = () => particle.remove();
                    }, delay + (Math.random() * 250)); 
                });

                setTimeout(() => { simBtn.disabled = false; }, 3200);
            });
        })();
    </script>
</div>
"""

display(HTML(html_code_conv))

# U-Net 1D

In [14]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from ddpm1d.ddpm1d_unet import DenoiseNet1D_UNet

device = torch.device("cpu")
n_T = 100
betas = np.linspace(1e-4, 0.02, n_T)

# Inicjalizacja modelu U-Net
model_unet = DenoiseNet1D_UNet(data_dim=128, time_emb_dim=64, base_channels=32).to(device)
ddpm_unet = DDPM1D(model=model_unet, betas=betas, n_T=n_T, device=device)

x_axis = np.linspace(0, 4 * np.pi, 128)
clean_signal = torch.tensor(np.sin(x_axis), dtype=torch.float32).unsqueeze(0).to(device)
fixed_noise = torch.randn_like(clean_signal)

def plot_unet_internals(t_val):
    model_unet.eval()
    
    with torch.no_grad():
        t_tensor = torch.tensor([t_val], device=device).long()
        alpha_bar_t = ddpm_unet.alphas_bar[t_tensor].view(-1, 1)
        
        if t_val == 0:
            x_input = clean_signal
        else:
            x_input = torch.sqrt(alpha_bar_t) * clean_signal + torch.sqrt(1 - alpha_bar_t) * fixed_noise
            
        # --- RĘCZNY FORWARD PASS DLA U-NET ---
        x_unsqueezed = x_input.unsqueeze(1)      # [1, 1, 128]
        t_emb = model_unet.time_embed(t_tensor)
        
        # Koder (w dół)
        x1 = model_unet.inc(x_unsqueezed)        # [1, 32, 128]
        x2_pool = model_unet.pool(x1)
        x2 = model_unet.down1(x2_pool, t_emb)    # [1, 64, 64]
        x3_pool = model_unet.pool(x2)
        x3 = model_unet.down2(x3_pool, t_emb)    # [1, 128, 32]
        
        # Wąskie gardło (Bottleneck)
        x_mid = model_unet.mid(x3, t_emb)        # [1, 128, 32]
        
        # Dekoder (w górę)
        x4_up = model_unet.up1(x_mid)            # [1, 128, 64]
        x4_cat = torch.cat([x4_up, x2], dim=1)   # [1, 192, 64] - SKIP CONNECTION
        x4 = model_unet.up_conv1(x4_cat, t_emb)  # [1, 64, 64]
        
        x5_up = model_unet.up2(x4)               # [1, 64, 128]
        x5_cat = torch.cat([x5_up, x1], dim=1)   # [1, 96, 128] - SKIP CONNECTION
        x5 = model_unet.up_conv2(x5_cat, t_emb)  # [1, 32, 128]
        
        out = model_unet.outc(x5).squeeze(1)     # [1, 128]

    tensors_to_plot = {
        f"1. Wejście: t={t_val} (1x128)": x_input.unsqueeze(0), 
        "2. Koder 'inc': 32 kanałów (32x128)": x1,
        "3. Koder 'down1': Zgniatanie w X, rośnie Y (64x64)": x2,
        "4. BOTTLENECK: Max kompresja (128x32)": x_mid,
        "5. Dekoder 'up_conv1' (po wklejeniu Skip Connection) (64x64)": x4,
        "6. Dekoder 'up_conv2' (powrót do oryginalnej długości) (32x128)": x5,
        "7. Wyjście: Przewidziany Szum (1x128)": out.unsqueeze(0)
    }

    fig, axes = plt.subplots(len(tensors_to_plot), 1, figsize=(10, 10), 
                             gridspec_kw={'height_ratios': [1, 2, 4, 8, 4, 2, 1]})
    fig.suptitle(f"Wnętrze U-Net 1D | Krok czasowy t = {t_val}", fontsize=14, fontweight='bold')

    for i, (title, tensor) in enumerate(tensors_to_plot.items()):
        ax = axes[i]
        data = tensor.squeeze(0).cpu().numpy() # [Kanały, Długość]
        
        im = ax.imshow(data, aspect='auto', cmap='plasma', interpolation='nearest')
        ax.set_title(title, fontsize=10, loc='left', pad=3)
        ax.set_yticks([]) 
        ax.set_xticks([]) 
        
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

t_slider_unet = widgets.IntSlider(value=99, min=0, max=n_T-1, step=1, description='Krok t:', continuous_update=False)
display(widgets.interactive(plot_unet_internals, t_val=t_slider_unet))

interactive(children=(IntSlider(value=99, continuous_update=False, description='Krok t:', max=99), Output()), …

In [15]:
from IPython.display import display, HTML

html_code_unet = """
<div style="border: 2px solid #ddd; padding: 15px; border-radius: 10px; background-color: #ffffff; font-family: sans-serif; width: 100%; max-width: 900px;">
    
    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px;">
        <div>
            <label style="font-weight: bold;">Krok czasowy t: <span id="t-value-unet">100</span></label><br>
            <input type="range" id="t-slider-unet" min="0" max="100" value="100" style="width: 200px;">
        </div>
        <button id="sim-btn-unet" style="padding: 10px 20px; font-size: 14px; font-weight: bold; cursor: pointer; background-color: #9C27B0; color: white; border: none; border-radius: 5px;">
            ▶ Symuluj U-Net 1D
        </button>
    </div>

    <div style="position: relative;">
        <svg id="nn-svg-unet" width="100%" height="520" style="background-color: #f8f9fa; border-radius: 5px; border: 1px solid #eee;"></svg>
    </div>

    <script>
        (function() {
            const svg = document.getElementById('nn-svg-unet');
            const slider = document.getElementById('t-slider-unet');
            const tValueDisplay = document.getElementById('t-value-unet');
            const simBtn = document.getElementById('sim-btn-unet');

            // Konstrukcja w kształcie litery 'U'
            // Oś Y reprezentuje kompresję wymiaru (L -> L/2 -> L/4)
            const layers = {
                input: Array.from({length: 8}, (_, i) => ({x: 40, y: 50 + i*20})),
                
                // Koder (w dół)
                inc: Array.from({length: 8}, (_, i) => ({x: 140, y: 80 + i*20})),     // L
                down1: Array.from({length: 4}, (_, i) => ({x: 260, y: 160 + i*20})),   // L/2
                down2: Array.from({length: 2}, (_, i) => ({x: 360, y: 240 + i*20})),   // L/4
                
                // Wąskie gardło (Bottleneck)
                mid: Array.from({length: 2}, (_, i) => ({x: 460, y: 240 + i*20})),     // L/4
                time: Array.from({length: 3}, (_, i) => ({x: 410, y: 380 + i*20})),    // Osadzenie Czasu
                
                // Dekoder (w górę)
                up1: Array.from({length: 4}, (_, i) => ({x: 560, y: 160 + i*20})),     // L/2
                up2: Array.from({length: 8}, (_, i) => ({x: 680, y: 80 + i*20})),      // L
                
                output: Array.from({length: 8}, (_, i) => ({x: 800, y: 50 + i*20}))
            };

            const connections = [];

            // Funkcja rysująca krawędzie
            function drawLink(source, target, type = "normal") {
                const line = document.createElementNS("http://www.w3.org/2000/svg", "line");
                line.setAttribute("x1", source.x); line.setAttribute("y1", source.y);
                line.setAttribute("x2", target.x); line.setAttribute("y2", target.y);
                
                if (type === "skip") {
                    line.setAttribute("stroke", "rgba(0, 188, 212, 0.4)"); // Błękitne mosty
                    line.setAttribute("stroke-dasharray", "4,4");
                    line.setAttribute("stroke-width", "2");
                } else if (type === "time") {
                    line.setAttribute("stroke", "rgba(255, 152, 0, 0.2)"); // Pomarańczowy czas
                    line.setAttribute("stroke-width", "1.5");
                } else {
                    line.setAttribute("stroke", "rgba(100, 100, 100, 0.2)");
                    line.setAttribute("stroke-width", "1");
                }
                
                svg.appendChild(line);
                connections.push({source, target, type});
            }

            // --- TWORZENIE SIATKI POŁĄCZEŃ ---
            
            // 1. Główny przepływ w kształcie U
            const connectBlocks = (layerA, layerB, prob=0.6) => {
                layerA.forEach(s => layerB.forEach(t => { if(Math.random() < prob) drawLink(s, t); }));
            };
            connectBlocks(layers.input, layers.inc, 1.0);
            connectBlocks(layers.inc, layers.down1);
            connectBlocks(layers.down1, layers.down2);
            connectBlocks(layers.down2, layers.mid, 1.0);
            connectBlocks(layers.mid, layers.up1);
            connectBlocks(layers.up1, layers.up2);
            connectBlocks(layers.up2, layers.output, 1.0);

            // 2. Skróty (Skip Connections)
            drawLink(layers.inc[0], layers.up2[0], "skip");
            drawLink(layers.inc[3], layers.up2[3], "skip");
            drawLink(layers.inc[7], layers.up2[7], "skip");
            
            drawLink(layers.down1[0], layers.up1[0], "skip");
            drawLink(layers.down1[3], layers.up1[3], "skip");

            // 3. Wstrzykiwanie Czasu (t_emb do każdego bloku UNetBlock1D)
            const connectTime = (targetLayer) => {
                layers.time.forEach(s => targetLayer.forEach(t => { if(Math.random() < 0.6) drawLink(s, t, "time"); }));
            };
            connectTime(layers.down1);
            connectTime(layers.down2);
            connectTime(layers.mid);
            connectTime(layers.up1);
            connectTime(layers.up2);


            // --- RYSOWANIE WĘZŁÓW ---
            function drawNode(x, y, color, r=5) {
                const circle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
                circle.setAttribute("cx", x); circle.setAttribute("cy", y);
                circle.setAttribute("r", r);
                circle.setAttribute("fill", color);
                circle.setAttribute("stroke", "#333");
                svg.appendChild(circle);
                return circle;
            }

            const inputNodes = layers.input.map(n => drawNode(n.x, n.y, "#03A9F4"));
            layers.inc.map(n => drawNode(n.x, n.y, "#9C27B0"));
            layers.down1.map(n => drawNode(n.x, n.y, "#7B1FA2"));
            layers.down2.map(n => drawNode(n.x, n.y, "#4A148C"));
            layers.mid.map(n => drawNode(n.x, n.y, "#E91E63"));
            layers.up1.map(n => drawNode(n.x, n.y, "#7B1FA2"));
            layers.up2.map(n => drawNode(n.x, n.y, "#9C27B0"));
            layers.output.map(n => drawNode(n.x, n.y, "#4CAF50"));
            layers.time.map(n => drawNode(n.x, n.y, "#FF9800"));

            // ETYKIETY
            const addText = (x, y, txt, bold=false) => {
                const t = document.createElementNS("http://www.w3.org/2000/svg", "text");
                t.setAttribute("x", x); t.setAttribute("y", y); t.setAttribute("font-size", "11");
                if (bold) t.setAttribute("font-weight", "bold");
                t.textContent = txt;
                svg.appendChild(t);
            };
            
            addText(20, 30, "Wejście (x)", true);
            addText(130, 60, "Koder (inc)", true);
            addText(240, 140, "down1 (L/2)");
            addText(340, 220, "down2 (L/4)");
            addText(445, 300, "Bottleneck (mid)", true);
            addText(550, 140, "up1 (L/2)");
            addText(670, 60, "up2 (L)", true);
            addText(780, 30, "Wyjście", true);
            addText(390, 460, "Czas (t_emb)", true);
            addText(380, 70, "Skip Connection (Kopia Sygnału)", true);

            // LOGIKA SUWAKA (Miganie szumu)
            slider.addEventListener('input', (e) => {
                const t = e.target.value;
                tValueDisplay.innerText = t;
                const noiseIntensity = t / 100; 
                inputNodes.forEach(node => {
                    const jitterX = 40 + (Math.random() - 0.5) * 15 * noiseIntensity;
                    node.setAttribute("cx", jitterX);
                    node.setAttribute("fill", `rgb(${Math.floor(noiseIntensity*200)}, 169, 244)`);
                });
            });

            // LOGIKA ANIMACJI KASKADOWEJ
            simBtn.addEventListener('click', () => {
                simBtn.disabled = true;
                
                connections.forEach((conn) => {
                    let delay = 0;
                    const sx = conn.source.x;
                    
                    // Definiujemy harmonogram wędrowania po literze U
                    if (conn.type === "normal") {
                        if (sx === 40) delay = 0;           // input -> inc
                        if (sx === 140) delay = 600;        // inc -> down1
                        if (sx === 260) delay = 1200;       // down1 -> down2
                        if (sx === 360) delay = 1800;       // down2 -> mid
                        if (sx === 460) delay = 2400;       // mid -> up1
                        if (sx === 560) delay = 3000;       // up1 -> up2
                        if (sx === 680) delay = 3600;       // up2 -> output
                    } 
                    else if (conn.type === "skip") {
                        // Kopia musi dolecieć na czas dekodowania!
                        if (sx === 140) delay = 2600; // inc -> up2 (doleci na 3200)
                        if (sx === 260) delay = 2000; // down1 -> up1 (doleci na 2600)
                    }
                    else if (conn.type === "time") {
                        // Czas uderza w konkretny blok w momencie jego aktywacji
                        const tx = conn.target.x;
                        if (tx === 260) delay = 600;  // uderza w down1
                        if (tx === 360) delay = 1200; // uderza w down2
                        if (tx === 460) delay = 1800; // uderza w mid
                        if (tx === 560) delay = 2400; // uderza w up1
                        if (tx === 680) delay = 3000; // uderza w up2
                    }

                    setTimeout(() => {
                        const particle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
                        particle.setAttribute("cx", conn.source.x);
                        particle.setAttribute("cy", conn.source.y);
                        particle.setAttribute("r", conn.type === "skip" ? 4 : 3);
                        
                        let pColor = "#03A9F4"; // Niebieski sygnał
                        if (conn.type === "time") pColor = "#FF9800"; // Pomarańczowy czas
                        if (conn.type === "skip") pColor = "#00BCD4"; // Błękitna kopia
                        particle.setAttribute("fill", pColor);
                        svg.appendChild(particle);

                        const animation = particle.animate([
                            { transform: `translate(0px, 0px)` },
                            { transform: `translate(${conn.target.x - conn.source.x}px, ${conn.target.y - conn.source.y}px)` }
                        ], {
                            duration: conn.type === "skip" ? 1000 : 600, // Skróty lecą trochę wolniej
                            easing: 'ease-in-out'
                        });

                        animation.onfinish = () => particle.remove();
                    }, delay + (Math.random() * 150)); 
                });

                // Odblokowanie po zakończeniu całej fali
                setTimeout(() => { simBtn.disabled = false; }, 4500);
            });
        })();
    </script>
</div>
"""

display(HTML(html_code_unet))